In [ ]:
"""
AI-Generated Email Evaluation System
Implements 6 evaluation metrics based on research framework
"""

# ============================================================================
# INSTALLATION & SETUP
# ============================================================================

# Install required packages
!pip install openai anthropic google-generativeai sentence-transformers scikit-learn pandas numpy tenacity -q tabulate

import os
import json
import time
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import openai
from anthropic import Anthropic
from tabulate import tabulate
from tenacity import retry, stop_after_attempt, wait_random_exponential

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for API keys and model selection"""

    # ============================================================================
    # OPENROUTER CONFIGURATION
    # ============================================================================
    # Set your OpenRouter API key here
    OPENROUTER_API_KEY = ""

    # OpenRouter base URL
    OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

    # Choose your model from OpenRouter's catalog
    EVALUATION_MODEL = "mistralai/ministral-8b"

    # Semantic entropy settings
    N_SEMANTIC_SAMPLES = 5  # Number of outputs to generate for semantic entropy

    # API Pricing (USD per 1M tokens) - OpenRouter pricing
    PRICING = {
    # OpenAI Models
    'openai/gpt-5': {'input': 1.25, 'output': 10.0},
    'openai/gpt-5-chat': {'input': 1.25, 'output': 10.0},
    'openai/gpt-4o': {'input': 2.5, 'output': 10.0},
    'openai/gpt-4-turbo': {'input': 10.0, 'output': 30.0},
    'openai/gpt-4o-mini': {'input': 0.15, 'output': 0.6},

    # Anthropic Models
    'anthropic/claude-sonnet-4': {'input': 3.0, 'output': 15.0},
    'anthropic/claude-3.5-haiku': {'input': 0.8, 'output': 4.0},
    'anthropic/claude-3-haiku': {'input': 0.25, 'output': 1.25},

    # Google Models
    'google/gemini-2.5-flash': {'input': 0.3, 'output': 2.5},
    'google/gemini-2.5-pro': {'input': 1.25, 'output': 10.0},
    'google/gemma-3-4b-it': {'input': 0.017, 'output': 0.068},

    # Qwen Models
    'qwen/qwen-2.5-72b-instruct': {'input': 0.35, 'output': 0.4},
    'qwen/qwen2.5-vl-72b-instruct': {'input': 0.0, 'output': 0.0},
    'qwen/qwen3-coder-30b-a3b-instruct': {'input': 0.06, 'output': 0.25},

    # Meta Models
    'meta-llama/llama-3.3-70b-instruct': {'input': 0.35, 'output': 0.4},
    'meta-llama/llama-3.1-8b-instruct': {'input': 0.016, 'output': 0.03},

    # Mistral Models
    'mistralai/mistral-small-3.2-24b-instruct:free': {'input': 0.0, 'output': 0.0},
    'mistralai/ministral-8b':{'input': 0.1, 'output': 0.1},
    'mistralai/mixtral-8x7b-instruct': {'input': 0.54, 'output': 0.54},

    # xAI Models
    'x-ai/grok-4-fast': {'input': 0.20, 'output': 0.50},

    # Amazon Models
    'amazon/nova-micro-1.0': {'input': 0.035, 'output': 0.14},

    # DeepSeek Models
    'deepseek/deepseek-v3-0324': {'input': 0.24, 'output': 0.84},
    'deepseek/deepseek-v3': {'input': 0.3, 'output': 0.85},

    # Llama Models:
    'sao10k/l3-lunaris-8b': {'input': 0.04, 'output': 0.05},
    }


    @classmethod
    def setup(cls):
        """Setup OpenRouter client"""
        # Configure OpenAI client to use OpenRouter
        client = openai.OpenAI(
            api_key=cls.OPENROUTER_API_KEY,
            base_url=cls.OPENROUTER_BASE_URL,
            default_headers={
                "HTTP-Referer": "https://github.com/yourusername/email-eval",
                "X-Title": "Email Evaluation System",
            }
        )
        return {
            'openai': client,
            'openrouter': client
        }

    @classmethod
    def get_model_cost(cls, model: str, input_tokens: int, output_tokens: int) -> float:
        """Calculate cost for a model call"""
        if model not in cls.PRICING:
            # If model not in pricing dict, return 0 (unknown cost)
            print(f"⚠️  Warning: No pricing info for model '{model}'. Cost tracking disabled.")
            return 0.0

        pricing = cls.PRICING[model]
        input_cost = (input_tokens / 1_000_000) * pricing['input']
        output_cost = (output_tokens / 1_000_000) * pricing['output']
        return input_cost + output_cost

# ============================================================================
# DATA STRUCTURES
# ============================================================================

@dataclass
class EvaluationResult:
    """Store evaluation results for a single email"""
    hallucination_score: int  # Binary: 0 or 1
    cta_quality: int  # 1-5
    language_quality: int  # 1-5
    personalization: int  # 1-5
    human_likeness: int  # 1-5
    instruction_adherence: int  # 1-5

    overall_score: float
    is_acceptable: bool
    detailed_feedback: Dict

    # Cost + latency tracking
    total_cost: float
    input_tokens: int
    output_tokens: int
    api_calls: int
    total_api_latency: float  # Sum of all LLM call durations (Layer 1 + 2)

    def to_dict(self):
        return {
            'hallucination_score': self.hallucination_score,
            'cta_quality': self.cta_quality,
            'language_quality': self.language_quality,
            'personalization': self.personalization,
            'human_likeness': self.human_likeness,
            'instruction_adherence': self.instruction_adherence,
            'overall_score': self.overall_score,
            'is_acceptable': self.is_acceptable,
            'total_cost_usd': self.total_cost,
            'input_tokens': self.input_tokens,
            'output_tokens': self.output_tokens,
            'api_calls': self.api_calls,
            'total_api_latency': self.total_api_latency,
            'detailed_feedback': self.detailed_feedback
        }

# ============================================================================
# LLM-AS-JUDGE EVALUATOR
# ============================================================================

class LLMEvaluator:
    """
    Uses LLM-as-judge approach for evaluating email quality
    Works with OpenRouter API
    """

    def __init__(self, model: str = Config.EVALUATION_MODEL):
        self.model = model
        self.clients = Config.setup()
        self.client = self.clients['openrouter']
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_api_calls = 0
        self.total_cost = 0.0
        self.email_api_latencies = []

    @retry(wait=wait_random_exponential(multiplier=1, max=60), stop=stop_after_attempt(3))
    def _call_llm(self, prompt: str, score_range: Tuple[int, int] = (1, 5)) -> Tuple[int, int, int]:
        """
        Call LLM via OpenRouter with retry mechanism
        Returns: (score, input_tokens, output_tokens)
        """

        try:
            t_api_start = time.perf_counter()
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Respond ONLY with a JSON object "
                            f"of the form {{\"score\": <integer between {score_range[0]} and {score_range[1]}>}}. "
                            "Do not add any other text or formatting."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0,
                max_tokens=10,
                response_format={"type": "json_object"},
            )
            t_api_end = time.perf_counter()
            self.email_api_latencies.append(t_api_end - t_api_start)

            input_tokens = response.usage.prompt_tokens
            output_tokens = response.usage.completion_tokens

            content = (response.choices[0].message.content or "").strip()
            if not content:
                print("⚠️ Empty response from model, returning -1")
                return -1, input_tokens, output_tokens

            try:
                data = json.loads(content)
            except json.JSONDecodeError:
                print(f"⚠️ JSON decode failed, raw content: {content}")
                return -1, input_tokens, output_tokens

            score = int(data.get("score", -1))

            # Cost tracking
            self.total_input_tokens += input_tokens
            self.total_output_tokens += output_tokens
            self.total_api_calls += 1
            self.total_cost += Config.get_model_cost(self.model, input_tokens, output_tokens)

            return score, input_tokens, output_tokens

        except Exception as e:
            print(f"❌ LLM call failed: {e}")
            raise

    def reset_usage(self):
        """Reset usage counters"""
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_api_calls = 0
        self.total_cost = 0.0

    def get_usage_stats(self) -> Dict:
        """Get current usage statistics"""
        return {
            'total_input_tokens': self.total_input_tokens,
            'total_output_tokens': self.total_output_tokens,
            'total_api_calls': self.total_api_calls,
            'total_cost_usd': round(self.total_cost, 8)
        }

    @staticmethod
    def safe_json_loads(text: str):
        """Robustly parse possibly wrapped JSON from model output."""
        import json, re

        if not text or not text.strip():
            raise ValueError("Empty LLM response")

        s = text.strip()

        # Remove ```json ... ``` wrapper
        if s.startswith("```"):
            s = s.strip("` \n")
            if s.lower().startswith("json"):
                s = s[4:].strip()

        # Try direct parsing
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass

        # Extract first { to last } and try again
        start, end = s.find("{"), s.rfind("}")
        if 0 <= start < end:
            snippet = s[start:end+1]
            try:
                return json.loads(snippet)
            except json.JSONDecodeError:
                pass

        # Replace single quotes with double quotes and retry
        s2 = re.sub(r"(?<!\\)'", '"', s)
        return json.loads(s2)

    def evaluate_hallucination(self, email: str, user_prompt: str) -> int:
        """Evaluate Hallucination (Binary Score: 0 or 1)"""
        prompt = f"""Evaluate the hallucination of this email on a binary scale of 0 or 1.

        Email:
        {email}

        {f"user_prompt: {user_prompt}" if user_prompt else ""}

        Description: A binary check for whether the model either fabricated information.

        Scoring:
        0: No Critical Failure. The output contains no fabricated information.
        1: Critical Failure Present. The output contains a clear and misleading falsehood that breaks trust. or has made up information not in the JSON.

        Respond only with a valid JSON object:
        {{
        "score": <integer 0 or 1>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(0, 1))
        return score

    def evaluate_cta_quality(self, email: str) -> int:
        """Evaluate Call-to-Action quality (1-5 scale)"""
        prompt = f"""Evaluate the Call-to-Action (CTA) quality of this email on a scale of 1-5.

        Email:
        {email}

        Evaluation Criteria:
        - Clarity: Is the desired action clear?
        - Relevance: Does it align with the email's purpose?
        - Effectiveness: Is it compelling and low-friction?
        - Appropriateness: Does it fit the context?

        Scoring:
        5: Excellent. The CTA is compelling, low-friction, contextually perfect, and directly supports the email's goal.
        4: Good. The CTA is clear and relevant but could be more compelling or better phrased.
        3: Average. The CTA is functional but generic or weak (e.g., "Let me know your thoughts").
        2: Poor. The CTA is vague, confusing, or mismatched with the email's tone/goal.
        1: Very Poor. The CTA is missing, inappropriate, or makes no sense.

        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score

    def evaluate_language_quality(self, email: str) -> int:
        """Evaluate language quality and coherence (1-5 scale)"""
        prompt = f"""Evaluate the language quality and structural coherence of this email on a scale of 1-5.

        Email:
        {email}

        Evaluation Criteria:
        - Grammar and spelling
        - Vocabulary appropriateness
        - Sentence structure variety
        - Conciseness (no unnecessary repetition)
        - Punctuation and formatting

        Scoring:
        5: Excellent - Free of errors with appropriate vocabulary, concise presentation, varied sentences, and proper formatting.
        4: Good - Mostly error-free with minor issues in vocabulary, conciseness, sentence variety, or formatting.
        3: Average - Some major errors present with noticeable issues in vocabulary, repetition, sentence variety, or formatting.
        2: Poor - Numerous errors with inappropriate vocabulary, significant repetition, limited sentence variety, and formatting issues.
        1: Very Poor - Pervasive errors that impede comprehension with inappropriate vocabulary, excessive repetition, and poor formatting.

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score

    def evaluate_personalization(self, email: str, recipient_data: Dict) -> int:
        """Evaluate personalization and personalization (1-5 scale)"""
        prompt = f"""Evaluate how well this email is tailored to the specific recipient and addresses their relevant needs.

        Email:
        {email}

        Recipient Data:
        {json.dumps(recipient_data, indent=2)}

        Evaluation Criteria (50% each):

        1. Recipient-Specific Tailoring
          - Uses at least 2 relevant data points (role, company, activities, signals).
          - Integrates details coherently (not a list or forced insertion).
          - Avoids prohibited info: founding year, employee count, work history >4 years.

        2. Relevance & Value
          - Addresses a real challenge or opportunity specific to the recipient's role.
          - Provides tangible value or actionable next step for the recipient.
          - Tone and timing fit their current context.

        Scoring guidelines:
        - 5: Highly personalized; clear unique value; deep understanding of recipient.
        - 4: Good personalization; relevant and useful.
        - 3: Basic personalization; somewhat generic but relevant.
        - 2: Minimal personalization; loosely relevant.
        - 1: Generic; no relevance or value.

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score

    def evaluate_human_likeness(self, email: str) -> int:
        """Evaluate how human-like the email sounds (1-5 scale)"""
        prompt = f"""Evaluate whether this email reads like it was written by a real person with natural thought patterns and authentic voice.

        Email:
        {email}

        Evaluation Criteria:

        - Naturalness: Conversational tone? Appropriate informality? Authentic voice?
        - Lexical Diversity: Balanced vocabulary richness? Natural word variety without forced sophistication?
        - Conciseness: Direct and efficient? Avoids unnecessary verbosity or over-explanation?
        - Emotional Authenticity: Genuine emotional expression? Appropriate sentiment depth and variety?
        - Structural Naturalness: Varied sentence structure? Natural flow without excessive complexity or formulaic patterns?

        Scoring:

        5: Highly Human-like - Conversational and authentic tone; balanced vocabulary with natural uniqueness; concise and direct; genuine emotional depth with varied sentiment; natural sentence variety and flow
        4: Mostly Human-like - Generally natural tone; good vocabulary balance; reasonably concise; authentic emotions present; varied structure with minor formulaic elements
        3: Ambiguous - Somewhat formal or generic tone; unbalanced vocabulary (too repetitive or artificially diverse); moderately verbose; limited emotional range; some overly complex or uniform sentences
        2: Likely AI-generated - Overly formal or polished tone; excessive use of sophisticated vocabulary or high repeatability; verbose with unnecessary elaboration; shallow or mismatched emotional expressions; complex sentence structures or excessive uniformity
        1: Clearly AI-generated - Robotic or excessively formal tone; unnatural vocabulary patterns (overuse of rare words or excessive repetition); extremely verbose with redundant content; generic or contextually inappropriate emotions; highly formulaic or unnaturally complex structures; lacks personal voice entirely

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score

    def evaluate_instruction_adherence(self, email: str, instructions: str, user_prompt: str) -> int:
        """Evaluate adherence to given instructions (1-5 scale)"""
        prompt = f"""Evaluate how well this email adheres to the given instructions.

        Email:
        {email}

        Instructions:
        {instructions}

        User Prompt:
        {user_prompt}

        Evaluation Criteria:
        - Completeness: Includes all required components
        - Accuracy: Follows instructions without deviation
        - Tone & Style: Maintains required tone
        - Alignment with Goals: Achieves intended purpose
        - Avoidance of Unnecessary Content: No irrelevant information

        Scoring:
        5: Excellent - Fully adheres with no deviations
        4: Good - Mostly adheres with minor omissions
        3: Average - Partially adheres, several deviations
        2: Poor - Frequently fails to follow key instructions
        1: Very Poor - Does not follow instructions at all

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score

# ============================================================================
# MAIN EVALUATION PIPELINE
# ============================================================================

class EmailEvaluationPipeline:
    """
    Complete evaluation pipeline for AI-generated emails
    Works with OpenRouter API
    """

    def __init__(self, model: str = Config.EVALUATION_MODEL):
        self.llm_eval = LLMEvaluator(model)
        self.clients = Config.setup()
        self.openrouter_client = self.clients['openrouter']

    # -------------------------------------------------------------
    # LAYER 1 — PRE-SCREENING
    # -------------------------------------------------------------
    def prescreen_email(self, email_id: int, email_text: str, user_prompt: str = "") -> Tuple[bool, Optional[str], Optional[int], List[float]]:
        """
        Layer 1 prescreen:
        - evaluate_hallucination -> binary (0/1)
        - evaluate_language_quality -> 1-5
        Returns:
            (passed: bool, reason: str_or_None, lang_score: int_or_None, prescreen_latencies: List[float])
        """
        prescreen_latencies: List[float] = []

        try:
            t0 = time.perf_counter()
            hi = self.llm_eval.evaluate_hallucination(email_text, user_prompt)
            prescreen_latencies.append(round(time.perf_counter() - t0, 3))
        except Exception as e:
            print(f"❌ Error during hallucination check for {email_id}: {e}")
            # Conservative approach: treat error as reject with reason
            return False, "hallucination_check_error", None, prescreen_latencies

        if hi == 1:
            return False, "hallucination_detected", None, prescreen_latencies

        try:
            t0 = time.perf_counter()
            lq = self.llm_eval.evaluate_language_quality(email_text)
            prescreen_latencies.append(round(time.perf_counter() - t0, 3))
        except Exception as e:
            print(f"❌ Error during language quality check for {email_id}: {e}")
            return False, "language_check_error", None, prescreen_latencies

        if lq < 3:
            return False, f"low_language_quality_{lq}", lq, prescreen_latencies

        # passed
        return True, None, lq, prescreen_latencies

    # -------------------------------------------------------------
    # LAYER 2 — FULL EVALUATION
    # -------------------------------------------------------------
    def evaluate_email(
        self,
        email: str,
        instructions: str = "",
        user_prompt: str = "",
        recipient_data: Dict = None,
        context: str = "",
        lang_score: int = None,
        hallucination_score: int = 0,
        prescreen_latencies: Optional[List[float]] = None,
    ) -> EvaluationResult:
        """
        Comprehensive email evaluation

        Args:
            email: The email text to evaluate
            instructions: Original instructions for email generation
            user_prompt: User instructions for email generation
            recipient_data: Dict with recipient information (role, company, etc.)
            context: Additional context for relevance evaluation
            lang_score: Precomputed language quality score from Layer 1
            hallucination_score: Precomputed hallucination score from Layer 1 (0 or 1)
        """

        start_time = time.time()
        print("Starting evaluation...")
        detailed_feedback = {}

        # Include Layer 1 latencies (hallucination + language) if provided
        email_api_latencies = list(prescreen_latencies or [])
        t_email_start = time.perf_counter()   # Record email evaluation start time

        # Reset usage counters for this evaluation
        self.llm_eval.reset_usage()

        # 1. CTA Quality
        print("Evaluating CTA quality...")
        t1 = time.perf_counter()
        cta_score = self.llm_eval.evaluate_cta_quality(email)
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 3))
        print(f"CTA score: {cta_score}")

        # 2. Personalization
        print("Evaluating personalization...")
        t1 = time.perf_counter()
        if recipient_data:
            per_score = self.llm_eval.evaluate_personalization(email, recipient_data)
        else:
            per_score = -1  # Default when no recipient data
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 3))
        print(f"Personalization score: {per_score}")

        # 3. Human-likeness
        print("Evaluating human-likeness...")
        t1 = time.perf_counter()
        hl_score = self.llm_eval.evaluate_human_likeness(email)
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 3))
        print(f"Human-likeness score: {hl_score}")

        # 4. Instruction Adherence
        print("Evaluating instruction adherence...")
        t1 = time.perf_counter()
        if instructions:
            ia_score = self.llm_eval.evaluate_instruction_adherence(email, instructions, user_prompt)
        else:
            ia_score = -1  # Default when no instruction
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 3))
        print(f"Instruction Adherence score: {ia_score}")

        # Get usage statistics
        usage_stats = self.llm_eval.get_usage_stats()

        # FIX: Properly handle language quality score
        if lang_score is None:
            # Defensive fallback (should not happen if pipeline used correctly)
            print("⚠️ Warning: lang_score not provided to evaluate_email(); calling evaluator as fallback.")
            t0 = time.perf_counter()
            lang_score = self.llm_eval.evaluate_language_quality(email)
            t1 = time.perf_counter()
            email_api_latencies.append(round(t1 - t0, 3))
            detailed_feedback["language_recomputed"] = True
        else:
            # Language score was precomputed in Layer 1
            detailed_feedback["language_precomputed"] = True

        # FIX: Calculate overall score correctly (exclude -1 invalid scores)
        valid_scores = []
        for score in [cta_score, lang_score, per_score, hl_score, ia_score]:
            if score != -1:  # Only include valid scores
                valid_scores.append(score)

        if valid_scores:
            overall_score = float(np.mean(valid_scores))
        else:
            overall_score = 0.0

        # FIX: Better acceptability judgment
        is_acceptable = (
            hallucination_score == 0 and
            overall_score >= 3 and
            lang_score >= 3
        )

        elapsed_time = time.time() - start_time
        detailed_feedback['runtime_seconds'] = elapsed_time
        t_email_end = time.perf_counter()
        detailed_feedback["email_total_time"] = t_email_end - t_email_start
        detailed_feedback["latencies"] = email_api_latencies
        detailed_feedback["total_api_latency"] = round(sum(email_api_latencies), 3)

        print(f"\nEvaluation complete!")
        print(f"💰 Cost: ${usage_stats['total_cost_usd']:.6f}")
        print(f"📊 Tokens: {usage_stats['total_input_tokens']} in / {usage_stats['total_output_tokens']:.6f} out")
        print(f"🔄 API Calls: {usage_stats['total_api_calls']}")
        print(f"⏱️ Runtime: {elapsed_time:.2f} seconds")

        return EvaluationResult(
            hallucination_score=hallucination_score,
            cta_quality=cta_score,
            language_quality=lang_score,
            personalization=per_score,
            human_likeness=hl_score,
            instruction_adherence=ia_score,
            overall_score=overall_score,
            is_acceptable=is_acceptable,
            detailed_feedback=detailed_feedback,
            total_cost=usage_stats['total_cost_usd'],
            input_tokens=usage_stats['total_input_tokens'],
            output_tokens=usage_stats['total_output_tokens'],
            api_calls=usage_stats['total_api_calls'],
            total_api_latency=detailed_feedback["total_api_latency"]
        )

    # -------------------------------------------------------------
    # LAYER 3 — BATCH EVALUATION
    # -------------------------------------------------------------
    def evaluate_batch(
        self,
        emails: List[Dict],
        save_rejected_path: str = "rejected_ids.json"
    ) -> pd.DataFrame:
        """
        Evaluate a list of emails.
        Each email dict should contain:
          - 'id' (optional)
          - 'email' (string)
          - 'instructions' (optional)
          - 'user_prompt' (optional)
          - 'recipient_data' (optional)
          - 'context' (optional)
        """

        pipeline_results = []
        rejected_records = []
        total_cost = 0.0
        batch_start = time.time()

        for idx, item in enumerate(emails):
            email_id = item.get("id", idx)
            email_text = item.get("email", "")
            instructions = item.get("instructions", "")
            user_prompt = item.get("user_prompt", "")
            recipient_data = item.get("recipient_data", {})
            context = item.get("context", "")

            print("\n" + "="*60)
            print(f"Processing email {idx+1}/{len(emails)} — id: {email_id}")
            print("="*60)

            # -------------------------
            # Layer 1: Pre-screening
            # -------------------------
            try:
                # FIX: Pass user_prompt parameter
                passed, reason, lang_score, prescreen_latencies = self.prescreen_email(
                    email_id=email_id,
                    email_text=email_text,
                    user_prompt=user_prompt  # FIX: Pass user_prompt for hallucination check
                )
            except Exception as e:
                print(f"❌ Prescreen error for {email_id}: {e}")
                passed = False
                reason = "prescreen_exception"
                lang_score = None
                prescreen_latencies = []

            if not passed:
                # record rejection and skip full evaluation
                print(f"❌ Rejected {email_id} during Layer 1: {reason}")
                rejected_records.append({"id": email_id, "reason": reason})
                continue

            # Passed prescreen; lang_score contains the language quality (1-5)
            print(f"✅ Passed prescreen. Language score: {lang_score}")

            # -------------------------
            # Layer 2: Full evaluation (reuse lang_score)
            # -------------------------
            try:
                result: EvaluationResult = self.evaluate_email(
                    email=email_text,
                    instructions=instructions,
                    user_prompt=user_prompt,
                    recipient_data=recipient_data,
                    context=context,
                    lang_score=lang_score,
                    hallucination_score=0,  # Layer 1 already checked, so hallucination score is 0
                    prescreen_latencies=prescreen_latencies
                )
            except Exception as e:
                print(f"❌ Error during full evaluation for {email_id}: {e}")
                # If full evaluation fails, record as rejected to be safe
                rejected_records.append({"id": email_id, "reason": "evaluation_failure"})
                continue

            pipeline_results.append({
                "email_id": email_id,
                **result.to_dict(),
                "precomputed_language_score": lang_score
            })

            total_cost += result.total_cost

        # Save rejected records to JSON
        try:
            with open(save_rejected_path, "w", encoding="utf-8") as f:
                json.dump(rejected_records, f, indent=2)
            print(f"\nSaved {len(rejected_records)} rejected emails → {save_rejected_path}")
        except Exception as e:
            print(f"❌ Failed to save rejected JSON: {e}")

        batch_elapsed = time.time() - batch_start
        print(f"\nBatch completed in {batch_elapsed:.2f}s — Evaluated {len(pipeline_results)} emails, Rejected {len(rejected_records)}")
        print(f"💰 Total estimated cost across evaluated emails: ${total_cost:.6f}")

        df = pd.DataFrame(pipeline_results)
        return df

# ============================================================================
# LAYER 3 — BUCKETING / AGGREGATION & RANKING
# ============================================================================

class Layer3Processor:
    """
    Layer 3: Bucketing, Aggregation & Ranking
    Classify, sort and summarize qualified emails from Layer 2 output
    """

    def __init__(self, bucket_config: Dict = None):
        """
        Initialize Layer 3 processor

        Args:
            bucket_config: Bucketing configuration, defaults to three-level bucketing
        """
        # Default bucket configuration
        self.default_bucket_config = {
            'high': {'min_score': 4.0, 'max_score': 5.0, 'label': 'High Score Emails (≥4.0)'},
            'medium': {'min_score': 3.0, 'max_score': 4.0, 'label': 'Medium Score Emails (3.0-4.0)'},
            'low': {'min_score': 0.0, 'max_score': 3.0, 'label': 'Low Score Emails (<3.0)'}
        }

        self.bucket_config = bucket_config or self.default_bucket_config

        # Sorting configuration
        self.sorting_config = {
            'default': ['overall_score', 'human_likeness', 'cta_quality'],
            'cta_priority': ['cta_quality', 'overall_score', 'human_likeness'],
            'human_priority': ['human_likeness', 'overall_score', 'cta_quality'],
            'language_priority': ['language_quality', 'overall_score', 'human_likeness'],
            'balanced': ['overall_score', 'language_quality', 'human_likeness', 'cta_quality', 'personalization', 'instruction_adherence']
        }

    def bucket_emails(self, df: pd.DataFrame, score_column: str = 'overall_score') -> Dict[str, pd.DataFrame]:
        """
        Bucket emails based on scores

        Args:
            df: DataFrame output from Layer 2
            score_column: Score column used for bucketing

        Returns:
            Dictionary: bucket name -> corresponding emails DataFrame
        """
        if df.empty:
            print("⚠️ Input DataFrame is empty, cannot perform bucketing")
            return {}

        bucketed_emails = {}

        for bucket_name, config in self.bucket_config.items():
            min_score = config['min_score']
            max_score = config['max_score']

            # Filter emails based on score range
            if max_score == 5.0:  # Highest bucket
                bucket_df = df[df[score_column] >= min_score].copy()
            else:
                bucket_df = df[(df[score_column] >= min_score) & (df[score_column] < max_score)].copy()

            if not bucket_df.empty:
                bucket_df['bucket'] = bucket_name
                bucket_df['bucket_label'] = config['label']
                bucketed_emails[bucket_name] = bucket_df

                print(f"📊 {config['label']}: {len(bucket_df)} emails")
            else:
                print(f"📊 {config['label']}: 0 emails")

        return bucketed_emails

    def rank_within_buckets(self,
                          bucketed_emails: Dict[str, pd.DataFrame],
                          sort_strategy: str = 'default',
                          ascending: bool = False) -> Dict[str, pd.DataFrame]:
        """
        Sort emails within each bucket

        Args:
            bucketed_emails: Bucketed emails dictionary
            sort_strategy: Sorting strategy ('default', 'cta_priority', 'human_priority', 'language_priority', 'balanced')
            ascending: Whether to sort in ascending order

        Returns:
            Sorted emails dictionary
        """
        if sort_strategy not in self.sorting_config:
            print(f"⚠️ Unknown sorting strategy '{sort_strategy}', using default strategy")
            sort_strategy = 'default'

        sort_columns = self.sorting_config[sort_strategy]

        ranked_buckets = {}

        for bucket_name, bucket_df in bucketed_emails.items():
            if bucket_df.empty:
                ranked_buckets[bucket_name] = bucket_df
                continue

            # Ensure sort columns exist
            available_columns = [col for col in sort_columns if col in bucket_df.columns]

            if not available_columns:
                print(f"⚠️ No available sort columns in bucket '{bucket_name}', using overall_score")
                available_columns = ['overall_score']

            # Sort
            sorted_df = bucket_df.sort_values(by=available_columns, ascending=ascending)

            # Add ranking
            sorted_df['bucket_rank'] = range(1, len(sorted_df) + 1)
            ranked_buckets[bucket_name] = sorted_df

        return ranked_buckets

    def generate_summary_statistics(self, df: pd.DataFrame, bucketed_emails: Dict[str, pd.DataFrame], total_emails_all: Optional[int] = None) -> Dict:
        """
        Generate summary statistics report

        Args:
            df: Original DataFrame
            bucketed_emails: Bucketed emails

        Returns:
            Dictionary containing various statistical information
        """
        if df.empty:
            return {}

        total_emails = total_emails_all if total_emails_all is not None else len(df)
        summary = {
            'total_emails': total_emails,
            'acceptable_emails': df['is_acceptable'].sum() if 'is_acceptable' in df.columns else None,
            'bucket_distribution': {},
            'score_statistics': {},
            'metric_statistics': {},
            'top_emails': {},
            'rejection_summary': {}
        }

        # Bucket distribution
        for bucket_name, bucket_df in bucketed_emails.items():
            bucket_label = self.bucket_config.get(bucket_name, {}).get('label', bucket_name)
            summary['bucket_distribution'][bucket_label] = {
                'count': len(bucket_df),
                'percentage': len(bucket_df) / len(df) * 100 if len(df) > 0 else 0
            }

        # Score statistics
        if 'overall_score' in df.columns:
            summary['score_statistics'] = {
                'mean': df['overall_score'].mean(),
                'median': df['overall_score'].median(),
                'std': df['overall_score'].std(),
                'min': df['overall_score'].min(),
                'max': df['overall_score'].max(),
                'q1': df['overall_score'].quantile(0.25),
                'q3': df['overall_score'].quantile(0.75)
            }

        # Metric statistics
        metrics = ['cta_quality', 'language_quality', 'personalization', 'human_likeness', 'instruction_adherence']
        for metric in metrics:
            if metric in df.columns:
                valid_scores = df[df[metric] >= 0][metric]  # Exclude -1 invalid scores
                if not valid_scores.empty:
                    summary['metric_statistics'][metric] = {
                        'mean': valid_scores.mean(),
                        'median': valid_scores.median(),
                        'min': valid_scores.min(),
                        'max': valid_scores.max(),
                        'count': len(valid_scores)
                    }

        # Top-3 emails per bucket
        for bucket_name, bucket_df in bucketed_emails.items():
            if not bucket_df.empty:
                top_3 = bucket_df.head(3)
                summary['top_emails'][bucket_name] = {
                    'count': len(top_3),
                    'emails': top_3[['email_id', 'overall_score'] + [m for m in metrics if m in top_3.columns]].to_dict('records')
                }

        # Rejected emails summary (if rejection reason exists)
        if 'rejection_reason' in df.columns:
            rejected_df = df[~df['rejection_reason'].isna() & (df['rejection_reason'] != '')]
            if not rejected_df.empty:
                rejection_counts = rejected_df['rejection_reason'].value_counts().to_dict()
                summary['rejection_summary'] = {
                    'total_rejected': len(rejected_df),
                    'rejection_reasons': rejection_counts
                }

        return summary

    # def get_top_n_emails(self,
    #                     ranked_buckets: Dict[str, pd.DataFrame],
    #                     n: int = 10,
    #                     strategy: str = 'balanced') -> pd.DataFrame:
    #     """
    #     Get global Top-N emails

    #     Args:
    #         ranked_buckets: Sorted bucketed emails
    #         n: Number of emails to retrieve
    #         strategy: Selection strategy ('balanced', 'high_only', 'proportional')

    #     Returns:
    #         DataFrame of Top-N emails
    #     """
    #     all_emails = []

    #     if strategy == 'high_only':
    #         # Only select from high bucket
    #         if 'high' in ranked_buckets:
    #             high_bucket = ranked_buckets['high']
    #             top_n = high_bucket.head(n)
    #             all_emails.append(top_n)
    #     elif strategy == 'proportional':
    #         # Select proportionally from each bucket
    #         total_count = sum(len(bucket_df) for bucket_df in ranked_buckets.values())
    #         for bucket_name, bucket_df in ranked_buckets.items():
    #             if total_count > 0:
    #                 proportion = len(bucket_df) / total_count
    #                 bucket_n = max(1, int(n * proportion))
    #                 all_emails.append(bucket_df.head(bucket_n))
    #     else:  # balanced (default)
    #         # Balanced selection: prioritize high bucket but include quality emails from other buckets
    #         selected_count = 0

    #         # First select from high bucket
    #         if 'high' in ranked_buckets and not ranked_buckets['high'].empty:
    #             high_selection = min(n // 2, len(ranked_buckets['high']))
    #             if high_selection > 0:
    #                 all_emails.append(ranked_buckets['high'].head(high_selection))
    #                 selected_count += high_selection

    #         # Select remaining spots from other buckets
    #         remaining_n = n - selected_count
    #         if remaining_n > 0:
    #             for bucket_name in ['medium', 'low']:
    #                 if bucket_name in ranked_buckets and not ranked_buckets[bucket_name].empty and remaining_n > 0:
    #                     bucket_selection = min(remaining_n // 2, len(ranked_buckets[bucket_name]))
    #                     if bucket_selection > 0:
    #                         all_emails.append(ranked_buckets[bucket_name].head(bucket_selection))
    #                         remaining_n -= bucket_selection

    #     if all_emails:
    #         result_df = pd.concat(all_emails, ignore_index=True)
    #         # Ensure no more than n emails
    #         result_df = result_df.head(n)
    #         return result_df
    #     else:
    #         return pd.DataFrame()

    def generate_report(self,
                      df: pd.DataFrame,
                      output_dir: str = "./layer3_reports",
                      top_n: int = 10,
                      total_emails_all: Optional[int] = None) -> Dict[str, any]:
        """
        Generate complete Layer 3 report

        Args:
            df: DataFrame output from Layer 2
            output_dir: Report output directory
            top_n: Number of Top-N emails to display

        Returns:
            Dictionary containing all report information
        """
        import os
        os.makedirs(output_dir, exist_ok=True)
        
        if df.empty:
            print("❌ No email data to process")
            return {}

        # Step 1: Bucketing
        print("\n1️⃣ Email Bucketing Results:")
        print("-" * 40)
        bucketed_emails = self.bucket_emails(df)

        # Step 2: Sort within buckets
        # print("\n2️⃣ In-bucket Sorting:")
        # print("-" * 40)
        ranked_buckets = self.rank_within_buckets(bucketed_emails, sort_strategy='balanced')

        # Step 3: Generate statistical summary
        # print("\n3️⃣ Statistical Summary:")
        # print("-" * 40)
        summary_stats = self.generate_summary_statistics(df, bucketed_emails, total_emails_all=total_emails_all)

        # Step 4: Get Top-N emails
        # print("\n4️⃣ Global Top-N Emails:")
        # print("-" * 40)
        #top_n_emails = self.get_top_n_emails(ranked_buckets, n=top_n, strategy='balanced')

        # Step 5: Generate detailed report
        print("\n2️⃣ Detailed Report:")
        print("-" * 40)

        # Save bucket data to CSV
        for bucket_name, bucket_df in ranked_buckets.items():
            bucket_label = self.bucket_config.get(bucket_name, {}).get('label', bucket_name)
            safe_label = bucket_label.replace(' ', '_').replace('(', '').replace(')', '').replace('<', 'lt').replace('≥', 'ge')
            csv_path = os.path.join(output_dir, f"{safe_label}.csv")
            bucket_df.to_csv(csv_path, index=False, encoding='utf-8')
            print(f"📁 {bucket_label} data saved to: {csv_path}")

        # Save Top-N emails
        # if not top_n_emails.empty:
        #     top_n_path = os.path.join(output_dir, f"top_{top_n}_emails.csv")
        #     top_n_emails.to_csv(top_n_path, index=False, encoding='utf-8')
        #     print(f"📁 Top-{top_n} emails saved to: {top_n_path}")

        # Save summary statistics as JSON
        if summary_stats:
            summary_path = os.path.join(output_dir, "summary_statistics.json")
            with open(summary_path, 'w', encoding='utf-8') as f:
                # Handle numpy types for JSON serialization
                import json

                def convert_to_serializable(obj):
                    if isinstance(obj, (np.integer, np.floating)):
                        return float(obj)
                    elif isinstance(obj, np.ndarray):
                        return obj.tolist()
                    elif isinstance(obj, pd.DataFrame):
                        return obj.to_dict('records')
                    else:
                        return obj

                serializable_summary = json.loads(json.dumps(summary_stats, default=convert_to_serializable))
                json.dump(serializable_summary, f, ensure_ascii=False, indent=2)
            print(f"📁 Summary statistics saved to: {summary_path}")

        # Print report
        self.print_report(summary_stats)

        # Return report data
        report = {
            'bucketed_emails': bucketed_emails,
            'ranked_buckets': ranked_buckets,
            'summary_stats': summary_stats,
            'output_dir': output_dir
        }

        return report


    def print_report(self, summary_stats: Dict):
        """
        Print report to console
        """
        print("\n" + "="*80)
        print("📈 Summary Statistics")
        print("="*80)

        if 'total_emails' in summary_stats:
            print(f"📊 Total emails: {summary_stats['total_emails']}")
            if summary_stats['acceptable_emails'] is not None:
                print(f"✅ Acceptable emails: {summary_stats['acceptable_emails']} ({summary_stats['acceptable_emails']/summary_stats['total_emails']*100:.1f}%)")

        print("\n📦 Bucket Distribution:")
        if 'bucket_distribution' in summary_stats:
            for bucket_label, stats in summary_stats['bucket_distribution'].items():
                print(f"  - {bucket_label}: {stats['count']} emails ({stats['percentage']:.1f}%)")

        print("\n🎯 Score Statistics:")
        if 'score_statistics' in summary_stats:
            stats = summary_stats['score_statistics']
            print(f"  - Mean: {stats['mean']:.2f}")
            print(f"  - Median: {stats['median']:.2f}")
            print(f"  - Standard deviation: {stats['std']:.2f}")
            print(f"  - Range: {stats['min']:.2f} - {stats['max']:.2f}")
            print(f"  - Interquartile range: Q1={stats['q1']:.2f}, Q3={stats['q3']:.2f}")

        print("\n📊 Metric Statistics:")
        if 'metric_statistics' in summary_stats:
            for metric, stats in summary_stats['metric_statistics'].items():
                metric_name = {
                    'cta_quality': 'CTA Quality',
                    'language_quality': 'Language Quality',
                    'personalization': 'Personalization',
                    'human_likeness': 'Human Likeness',
                    'instruction_adherence': 'Instruction Adherence'
                }.get(metric, metric)
                print(f"  - {metric_name}: {stats['mean']:.2f} (range: {stats['min']:.2f}-{stats['max']:.2f}, samples: {stats['count']})")

        if 'rejection_summary' in summary_stats and summary_stats['rejection_summary']:
            print("\n❌ Rejected Emails Summary:")
            rej = summary_stats['rejection_summary']
            print(f"  - Total rejected: {rej['total_rejected']}")
            for reason, count in rej['rejection_reasons'].items():
                print(f"    - {reason}: {count}")

        # print(f"\n🏆 Top-{top_n} Recommended Emails:")
        # print("="*80)
        # if not top_n_emails.empty:
        #     # Select columns to display
        #     display_columns = ['email_id', 'overall_score', 'bucket_label', 'bucket_rank']
        #     metrics = ['cta_quality', 'human_likeness', 'language_quality', 'personalization', 'instruction_adherence']

        #     for col in metrics:
        #         if col in top_n_emails.columns:
        #             display_columns.append(col)

        #     display_df = top_n_emails[display_columns].copy()

        #     # Rename columns for display
        #     column_names = {
        #         'email_id': 'Email ID',
        #         'overall_score': 'Overall Score',
        #         'bucket_label': 'Bucket',
        #         'bucket_rank': 'Bucket Rank',
        #         'cta_quality': 'CTA Quality',
        #         'human_likeness': 'Human Likeness',
        #         'language_quality': 'Language Quality',
        #         'personalization': 'Personalization',
        #         'instruction_adherence': 'Instruction Adherence'
        #     }

        #     display_df = display_df.rename(columns=column_names)
        #     # ✅ MODIFIED: Use tabulate for prettier output
        #     print(tabulate(display_df,
        #                   headers='keys',
        #                   tablefmt='grid',
        #                   showindex=False,
        #                   numalign='center',
        #                   stralign='center'))
        # else:
        #     print("❌ No 
            
            
        #     ed emails")

        print("\n" + "="*80)
        print("📋 Report generation completed")
        print("="*80)

# ============================================================================
# COMPLETE EVALUATION PIPELINE
# ============================================================================

class CompleteEvaluationPipeline:
    """
    Complete evaluation pipeline: Full process including Layer 1-2-3
    """

    def __init__(self, model: str = Config.EVALUATION_MODEL):
        self.layer12_pipeline = EmailEvaluationPipeline(model=model)
        self.layer3_processor = Layer3Processor()
        self.total_cost = 0.0
        self.total_time = 0.0

    def run_complete_evaluation(self,
                              emails: List[Dict],
                              save_rejected_path: str = "rejected_ids.json",
                              output_dir: str = "./evaluation_reports",
                              top_n: int = 10) -> Dict[str, any]:
        """
        Run complete evaluation process

        Args:
            emails: List of emails
            save_rejected_path: Path to save rejected emails
            output_dir: Output directory
            top_n: Number of Top-N emails

        Returns:
            Dictionary containing all results
        """
        import time
        start_time = time.time()

        print("\n" + "="*80)
        print("🚀 Starting complete email evaluation process (Layer 1-2-3)")
        print("="*80)

        # Layer 1-2: Email evaluation
        print("\n📋 Layer 1-2: Evaluating emails...")
        print("-" * 40)

        try:
            df_layer2 = self.layer12_pipeline.evaluate_batch(
                emails=emails,
                save_rejected_path=save_rejected_path
            )

            if df_layer2.empty:
                print("❌ No emails passed Layer 1-2 evaluation")
                return {}

            self.total_cost = df_layer2['total_cost_usd'].sum() if 'total_cost_usd' in df_layer2.columns else 0.0

        except Exception as e:
            print(f"❌ Layer 1-2 evaluation failed: {e}")
            return {}

        # Layer 3: Bucketing, sorting, summarization
        print("\n📊 Layer 3: Bucketing, sorting and summarizing...")
        print("-" * 40)

        try:
            # Ensure necessary columns exist
            if 'overall_score' not in df_layer2.columns:
                print("❌ Layer 2 output missing overall_score column")
                return {}

            # Run Layer 3
            layer3_report = self.layer3_processor.generate_report(
                df=df_layer2,
                output_dir=output_dir,
                top_n=top_n,
                total_emails_all=len(emails)
            )

        except Exception as e:
            print(f"❌ Layer 3 processing failed: {e}")
            import traceback
            traceback.print_exc()
            layer3_report = {}

        # Calculate total time
        self.total_time = time.time() - start_time

        # Final summary
        print("\n" + "="*80)
        print("🎉 Complete evaluation process finished")
        print("="*80)
        print(f"⏱️  Total time: {self.total_time:.2f} seconds")
        print(f"💰 Total cost: ${self.total_cost:.6f}")
        print(f"📁 Reports saved to: {output_dir}")
        print("="*80)

        # Return complete results
        full_results = {
            'layer2_results': df_layer2,
            'layer3_report': layer3_report,
            'total_cost': self.total_cost,
            'total_time': self.total_time,
            'output_dir': output_dir
        }

        return full_results

In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_test_data_from_file(file_path: str) -> List[Dict]:
    """
    Load test data from an external file.

    Supported formats:
    - JSON (.json): a JSON array containing test data

    Args:
        file_path: Path to the data file

    Returns:
        List of test data dictionaries
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ File not found: {file_path}")

    file_ext = os.path.splitext(file_path)[1].lower()

    try:
        if file_ext == '.json':
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        else:
            raise ValueError(f"❌ Unsupported file format: {file_ext}. Supported format: .json")

        # Validate data structure
        required_fields = ['email']
        for i, item in enumerate(data):
            for field in required_fields:
                if field not in item:
                    raise ValueError(f"❌ Missing required field '{field}' in item {i}")

        print(f"✅ Successfully loaded {len(data)} test samples")
        return data

    except Exception as e:
        print(f"❌ Failed to load test data: {e}")
        raise

# ============================================================================
# MAIN TEST FUNCTIONS
# ============================================================================

def test_spreadsheet_examples(file_path: str):
    """Run evaluation pipeline on external test data from a JSON file"""

    print("\n" + "="*80)
    print("🧪 TESTING WITH EXTERNAL TEST DATA")
    print("="*80)

    # Load test data
    print(f"📁 Loading test data from file: {file_path}")
    test_data = load_test_data_from_file(file_path)

    if not test_data:
        print("❌ No test data found")
        return None

    # Initialize the evaluation pipeline
    try:
        pipeline = EmailEvaluationPipeline(model=Config.EVALUATION_MODEL)
    except Exception as e:
        print(f"\n❌ Error: Unable to initialize evaluation pipeline - {e}")
        return None

    # Use correct two-layer evaluation process
    results_list = []
    rejected_records = []  # Record rejected emails
    total_hallucination_detected = 0  # Record hallucination detection count

    for idx, example in enumerate(test_data):
        example_id = example.get('id', idx)
        print(f"\n{'='*80}")
        print(f"📧 EVALUATING EXAMPLE {example_id}")
        recipient_data = example.get('recipient_data', {})
        print(f"Recipient: {recipient_data.get('name', 'Unknown')} - {recipient_data.get('role', 'Unknown')}")
        print(f"{'='*80}")

        # Print the email being evaluated
        print("\n📧 EMAIL BEING EVALUATED:")
        print("-" * 80)
        print(example['email'])
        print("-" * 80)

        # First run Layer 1 pre-screening
        print("\n🔍 LAYER 1: PRE-SCREENING")
        print("-" * 40)

        try:
            user_prompt = example.get('user_prompt', '')
            # Call prescreen_email for Layer 1 check
            passed, reason, lang_score, prescreen_latencies = pipeline.prescreen_email(
                email_id=example_id,
                email_text=example['email'],
                user_prompt=user_prompt
            )

            if not passed:
                print(f"❌ Email {example_id} rejected in Layer 1: {reason}")
                rejected_records.append({
                    'email_id': example_id,  # Use 'email_id' instead of 'example_id'
                    'reason': reason,
                    'language_score': lang_score if lang_score else -1
                })

                # If hallucination, record details
                if reason == "hallucination_detected":
                    total_hallucination_detected += 1

                continue

            print(f"✅ Email {example_id} passed Layer 1")
            print(f"📊 Language quality score from Layer 1: {lang_score}")

        except Exception as e:
            print(f"❌ Layer 1 pre-screening failed for email {example_id}: {e}")
            rejected_records.append({
                'email_id': example_id,  # Use 'email_id' instead of 'example_id'
                'reason': f"Layer 1 error: {str(e)}",
                'language_score': -1
            })
            continue

        # Only run Layer 2 full evaluation for emails that passed Layer 1
        print("\n📊 LAYER 2: FULL EVALUATION")
        print("-" * 40)

        try:
            result = pipeline.evaluate_email(
                email=example['email'],
                instructions=example.get('instructions', ''),
                user_prompt=user_prompt,
                recipient_data=recipient_data,
                context=example.get('context', ''),
                lang_score=lang_score,  # Pass language score calculated in Layer 1
                hallucination_score=0,   # Layer 1 already passed, so hallucination score is 0
                prescreen_latencies=prescreen_latencies
            )

            results_list.append({
                'email_id': example_id,  # FIX: Use 'email_id' instead of 'example_id'
                'overall_score': result.overall_score,
                'is_acceptable': result.is_acceptable,  # FIX: Use 'is_acceptable' to match Layer3Processor
                'hallucination': result.hallucination_score,  # Should be 0 (already checked)
                'cta_quality': result.cta_quality,
                'language_quality': result.language_quality,
                'personalization': result.personalization,
                'human_likeness': result.human_likeness,
                'instruction_adherence': result.instruction_adherence,
                'total_cost_usd': result.total_cost,  # FIX: Use 'total_cost_usd' to match Layer3Processor
                'total_api_latency': result.total_api_latency,
                'latencies': result.detailed_feedback.get("latencies"),
                'email_total_time': result.detailed_feedback.get("email_total_time"),
                'rejection_reason': ""
            })

            print(f"✅ Email {example_id} evaluation completed successfully")
            print(f"📊 Overall score: {result.overall_score:.2f}/5")

        except Exception as e:
            print(f"❌ Layer 2 evaluation failed for email {example_id}: {e}")
            rejected_records.append({
                'email_id': example_id,  # Use 'email_id' instead of 'example_id'
                'reason': f"Layer 2 error: {str(e)}",
                'language_score': lang_score
            })

    # Summary
    print("\n\n" + "="*80)
    print("📊 TEST RESULTS SUMMARY")
    print("="*80)

    # Show detailed rejection statistics
    total_emails = len(test_data)
    evaluated_emails = len(results_list)
    rejected_emails = len(rejected_records)

    print(f"📧 Total Emails: {total_emails}")
    print(f"✅ Evaluated Emails (Passed Layer 1): {evaluated_emails}")
    print(f"❌ Rejected Emails: {rejected_emails}")

    if rejected_emails > 0:
        print("\n📋 REJECTION DETAILS:")
        rejection_reasons = {}
        for record in rejected_records:
            reason = record['reason']
            rejection_reasons[reason] = rejection_reasons.get(reason, 0) + 1

        for reason, count in rejection_reasons.items():
            print(f"  - {reason}: {count} emails")


    if results_list:
        # Add display-friendly cost string to accepted rows
        for item in results_list:
            if 'total_cost_usd' in item:
                item['total_cost_display'] = f"${item['total_cost_usd']:.8f}"

        df = pd.DataFrame(results_list)
        pd.set_option('display.float_format', lambda x: f'{x:.8f}')
        print("\n📈 EVALUATED EMAILS STATISTICS:")
        print("-" * 40)

        display_columns = ['email_id', 'overall_score', 'is_acceptable', 'hallucination',
                           'cta_quality', 'personalization', 'human_likeness', 'language_quality',
                           'instruction_adherence', 'total_cost_display', 'total_api_latency', 'latencies', 'email_total_time']


        display_df = df[display_columns].copy()
        column_names = {
            'email_id': 'Email ID',
            'overall_score': 'Overall Score',
            'total_cost_display': 'Cost (USD)',
            'is_acceptable': 'Acceptable',
            'cta_quality': 'CTA Quality',
            'human_likeness': 'Human Likeness',
            'language_quality': 'Language Quality',
            'personalization': 'Personalization',
            'instruction_adherence': 'Instruction Adherence',
            'total_api_latency': 'Total API Latency (s)',
            'latencies': 'API Latencies',
            'email_total_time': 'Email Total Time'

        }
        display_df = display_df.rename(columns={k: v for k, v in column_names.items() if k in display_df.columns})


        print(tabulate(display_df,
                      headers='keys',
                      tablefmt='grid',
                      showindex=False,
                      stralign='center'))

        total_cost = df['total_cost_usd'].sum()
        avg_score = df['overall_score'].mean()
        acceptable_count = df['is_acceptable'].sum()  # Use 'is_acceptable'

        # Correct score range display
        print(f"\n{'='*80}")
        print(f"💰 Total Cost for Evaluated Emails: ${total_cost:.8f}")
        print(f"💰 Average Cost per Evaluated Email: ${total_cost/evaluated_emails:.8f}")
        pd.reset_option('display.float_format')
        print(f"💰 Estimated Cost for 10,000 similar emails: ${(total_cost/evaluated_emails)*10000:.2f}")

        # Global latency stats (only for evaluated emails)
        if hasattr(pipeline.llm_eval, 'email_api_latencies') and pipeline.llm_eval.email_api_latencies:
            lat_arr = np.array(pipeline.llm_eval.email_api_latencies)
            print("\n⏱️ API Latency Distribution (ALL CALLS)")
            print(f"  P50: {np.percentile(lat_arr, 50):.4f} s")
            print(f"  P95: {np.percentile(lat_arr, 95):.4f} s")
            print(f"  P99: {np.percentile(lat_arr, 99):.4f} s")

        # Total batch time
        total_runtime = df['email_total_time'].sum()
        print(f"\n🕒 Total Batch Runtime: {total_runtime:.2f} s")

        # Correct score range display
        print(f"\n📈 Average Overall Score: {avg_score:.2f}/5")
        print(f"✅ Acceptable Emails: {acceptable_count}/{total_emails} ({acceptable_count/total_emails*100:.0f}%)")

        # Save results
        output_file = "evaluation_results.csv"
        df.to_csv(output_file, index=False)
        print(f"\n📁 Results saved to: {output_file}")

        # Optionally run Layer 3 analysis
        if evaluated_emails > 0:
            print("\n" + "="*80)
            print("📊 LAYER 3: BUCKETING & RANKING")
            print("="*80)

            try:
                # Initialize Layer 3 processor
                layer3_processor = Layer3Processor()

                # Generate Layer 3 report
                report = layer3_processor.generate_report(
                    df=df,
                    output_dir="./test_results_layer3",
                    top_n=min(10, evaluated_emails)  # Show Top-N, but not more than data available
                )

                print("✅ Layer 3 analysis completed successfully")
            except Exception as e:
                print(f"⚠️ Layer 3 analysis failed: {e}")
                import traceback
                traceback.print_exc()

        print(f"{'='*80}\n")
        return df
    else:
        print("❌ No emails passed Layer 1 evaluation")
        return pd.DataFrame()

def analyze_specific_example(file_path: str, example_id: int = 0):
    """Run detailed evaluation on a specific example"""

    print("\n" + "="*80)
    print(f"🔍 DETAILED ANALYSIS: EXAMPLE {example_id}")
    print("="*80)

    # Load test data
    test_data = load_test_data_from_file(file_path)

    # Find the specific example
    example = None
    for item in test_data:
        if item.get('id') == example_id:
            example = item
            break

    if not example:
        print(f"❌ Example with ID {example_id} not found")
        return None

    print("\n📧 EMAIL BEING EVALUATED:")
    print("-" * 80)
    print(example['email'])
    print("-" * 80)

    print("\n📋 EVALUATION CONTEXT:")
    recipient_data = example.get('recipient_data', {})
    print(f"Recipient: {recipient_data.get('name', 'Unknown')}")
    print(f"Role: {recipient_data.get('role', 'Unknown')}")
    print(f"Company: {recipient_data.get('company', 'Unknown')}")
    print(f"Context: {example.get('context', 'No context')}")

    print("\n📝 INSTRUCTIONS GIVEN:")
    print(example.get('instructions', 'No instructions provided'))

    pipeline = EmailEvaluationPipeline(model=Config.EVALUATION_MODEL)

    # Use correct two-layer evaluation process
    print("\n🔍 LAYER 1: PRE-SCREENING")
    print("-" * 40)

    user_prompt = example.get('user_prompt', '')
    passed, reason, lang_score, prescreen_latencies = pipeline.prescreen_email(
        email_id=example_id,
        email_text=example['email'],
        user_prompt=user_prompt
    )

    if not passed:
        print(f"❌ Email {example_id} rejected in Layer 1: {reason}")
        return None

    print(f"✅ Email {example_id} passed Layer 1")
    print(f"📊 Language quality score: {lang_score}")

    # Correctly call evaluate_email
    result = pipeline.evaluate_email(
        email=example['email'],
        instructions=example.get('instructions', ''),
        user_prompt=user_prompt,
        recipient_data=recipient_data,
        context=example.get('context', ''),
        lang_score=lang_score,
        hallucination_score=0,
        prescreen_latencies=prescreen_latencies
    )

    print("\n" + "="*80)
    print("📊 EVALUATION RESULTS")
    print("="*80)
    print(f"Overall Score: {result.overall_score:.1f}/5")  # Correct score range
    print(f"Acceptable: {'✅ YES' if result.is_acceptable else '❌ NO'}")
    print(f"\nBreakdown:")
    print(f"  Hallucination: {result.hallucination_score}/1")  # Show hallucination score
    print(f"  CTA Quality: {result.cta_quality}/5")
    print(f"  Language Quality: {result.language_quality}/5")
    print(f"  Personalization: {result.personalization}/5")
    print(f"  Human-likeness: {result.human_likeness}/5")
    print(f"  Instruction Adherence: {result.instruction_adherence}/5")
    print(f"\n💰 Cost: ${result.total_cost:.6f}")

    print("\n" + "="*80)
    print("💬 DETAILED FEEDBACK")
    print("="*80)
    for criterion, feedback in result.detailed_feedback.items():
        print(f"\n{criterion.upper()}:")
        if isinstance(feedback, dict):
            for key, value in feedback.items():
                print(f"  {key}: {value}")
        elif isinstance(feedback, list):
            print(f"  {feedback}")
        else:
            print(f"  {feedback}")

    return result

# ============================================================================
# NEW: Test function using complete three-layer pipeline
# ============================================================================

def test_complete_pipeline(file_path: str):
    """Test using complete three-layer pipeline"""

    print("\n" + "="*80)
    print("🚀 TESTING COMPLETE 3-LAYER PIPELINE")
    print("="*80)

    # Load test data
    test_data = load_test_data_from_file(file_path)

    if not test_data:
        print("❌ No test data found")
        return None

    # Initialize complete pipeline
    try:
        complete_pipeline = CompleteEvaluationPipeline(model=Config.EVALUATION_MODEL)
    except Exception as e:
        print(f"❌ Failed to initialize CompleteEvaluationPipeline: {e}")
        return None

    # Run complete evaluation
    results = complete_pipeline.run_complete_evaluation(
        emails=test_data,
        output_dir="./complete_evaluation_results",
        top_n=10
    )

    return results

# ============================================================================
# MAIN EXECUTION ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    print("\n🚀 Starting Email Evaluation System...")

    # Hardcoded file path for Colab
    file_path = "/content/email_100.json"

    # Provide multiple run modes
    print("\n🔧 AVAILABLE MODES:")
    print("1. Layer 1-2 Test (Corrected)")
    print("2. Complete 3-Layer Pipeline")
    print("3. Analyze specific example")

    # Choose run mode
    MODE = 1  # 1, 2, or 3

    if MODE == 1:
        print(f"\n🧪 Running Layer 1-2 Test from: {file_path}")
        results_df = test_spreadsheet_examples(file_path=file_path)
    elif MODE == 2:
        print(f"\n🧪 Running COMPLETE 3-Layer Pipeline from: {file_path}")
        results = test_complete_pipeline(file_path=file_path)
    elif MODE == 3:
        EXAMPLE_ID = 0  # Set email ID to analyze
        print(f"\n🔍 Running detailed analysis on example ID {EXAMPLE_ID}")
        analyze_specific_example(file_path=file_path, example_id=EXAMPLE_ID)
    else:
        print("❌ Invalid mode selected")


🚀 Starting Email Evaluation System...

🔧 AVAILABLE MODES:
1. Layer 1-2 Test (Corrected)
2. Complete 3-Layer Pipeline
3. Analyze specific example

🧪 Running Layer 1-2 Test from: /content/20_layer_test_emails.json

🧪 TESTING WITH EXTERNAL TEST DATA
📁 Loading test data from file: /content/20_layer_test_emails.json
✅ Successfully loaded 20 test samples

📧 EVALUATING EXAMPLE 1
Recipient: Unknown - Unknown

📧 EMAIL BEING EVALUATED:
--------------------------------------------------------------------------------
Hey Damilola,  

Nurse shortages and heavy workloads make it harder for teams to give timely care. Chasing providers or waiting for callbacks only adds delays and stress for everyone involved.  

TigerConnect removes that wait by routing messages instantly to the right provider. It helps nurses connect fast, so patients are cared for sooner and staff feel less pressure.  

Is improving this kind of communication a focus for you right now? We could set up a call to discuss how it migh